In [32]:
import pandas as pd
import numpy as np
from scipy.stats import gmean
pd.set_option("display.max_columns", None)

In [33]:
df = pd.read_csv("bien_traits_summary.csv")
df.head(20)

/var/folders/h6/qd2kpqfj4_35f63hmcw_m0nc0000gn/T/ipykernel_4170/1996813199.py:1: DtypeWarning: Columns (2,4,8,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("bien_traits_summary.csv")


,scrubbed_species_binomial,trait_name,trait_value,unit,method,latitude,longitude,elevation_m,url_source,project_pi,project_pi_contact,access,id
0,Fraxinus americana,whole plant height,25.6032,m,NaN,43.810731,-71.470813,NaN,NaN,Greg Reams,NaN,public,26838000.0
1,Quercus rubra,whole plant height,21.336,m,NaN,43.637349,-72.083976,NaN,NaN,Greg Reams,NaN,public,26838033.0
2,Quercus rubra,whole plant height,23.4696,m,NaN,44.401370,-71.099594,NaN,NaN,Greg Reams,NaN,public,26838058.0
3,Quercus rubra,whole plant height,8.5344,m,NaN,44.064089,-71.960788,NaN,NaN,Greg Reams,NaN,public,26838073.0
4,Quercus rubra,whole plant height,11.8872,m,NaN,42.990632,-71.274879,NaN,NaN,Greg Reams,NaN,public,26838050.0
5,Fraxinus americana,whole plant height,22.86,m,NaN,43.883496,-71.743833,NaN,NaN,Greg Reams,NaN,public,26838064.0
6,Quercus rubra,whole plant height,24.384,m,NaN,43.169580,-71.947917,NaN,NaN,Greg Reams,NaN,public,26838077.0
7,Quercus rubra,whole plant height,21.9456,m,NaN,42.746245,-71.460145,NaN,NaN,Greg Reams,NaN,public,26838078.0
8,Fraxinus americana,whole plant height,20.1168,m,NaN,43.143692,-72.104246,NaN,NaN,Greg Reams,NaN,public,26838020.0
9,Fraxinus nigra,whole plant height,11.5824,m,NaN,44.354441,-71.530688,NaN,NaN,Greg Reams,NaN,public,26838024.0


In [35]:
unique_vals = {
    col: df[col].unique()
    for col in df.columns
}
unique_vals

{'scrubbed_species_binomial': array(['Fraxinus americana', 'Quercus rubra', 'Fraxinus nigra',
        'Picea mariana', 'Quercus alba', 'Liquidambar styraciflua',
        'Quercus phellos', 'Magnolia virginiana', 'Quercus bicolor',
        'Pseudotsuga menziesii', 'Picea engelmannii', 'Pinus taeda',
        'Ulmus americana', 'Fraxinus pennsylvanica', 'Picea glauca',
        'Populus deltoides', 'Juglans cinerea', 'Juglans nigra',
        'Fraxinus quadrangulata', 'Cercis canadensis', 'Cornus florida',
        'Thuja plicata', 'Picea sitchensis', 'Umbellularia californica',
        'Pinus albicaulis', 'Pinus lambertiana', 'Inga laurina',
        'Castanea dentata', 'Carya illinoinensis', 'Pinus radiata',
        'Quercus lobata', 'Quercus agrifolia', 'Quercus wislizeni',
        'Sequoiadendron giganteum', 'Melaleuca quinquenervia',
        'Malus fusca', 'Laguncularia racemosa', 'Trichilia hirta',
        'Senna siamea', 'Erythroxylum rotundifolium', 'Pimenta racemosa',
        'Cedrel

In [36]:
traits = df["trait_name"].unique()

trait_dfs = {
    trait: df[df["trait_name"] == trait].copy()
    for trait in traits
}
traits

array(['whole plant height', 'whole plant growth form', 'leaf area',
       'stem wood density', 'seed mass', 'whole plant sexual system',
       'flower pollination syndrome', 'fruit type'], dtype=object)

In [37]:
trait_dfs["whole plant height"]

,scrubbed_species_binomial,trait_name,trait_value,unit,method,latitude,longitude,elevation_m,url_source,project_pi,project_pi_contact,access,id
0,Fraxinus americana,whole plant height,25.6032,m,NaN,43.810731,-71.470813,NaN,NaN,Greg Reams,NaN,public,26838000.0
1,Quercus rubra,whole plant height,21.336,m,NaN,43.637349,-72.083976,NaN,NaN,Greg Reams,NaN,public,26838033.0
2,Quercus rubra,whole plant height,23.4696,m,NaN,44.401370,-71.099594,NaN,NaN,Greg Reams,NaN,public,26838058.0
3,Quercus rubra,whole plant height,8.5344,m,NaN,44.064089,-71.960788,NaN,NaN,Greg Reams,NaN,public,26838073.0
4,Quercus rubra,whole plant height,11.8872,m,NaN,42.990632,-71.274879,NaN,NaN,Greg Reams,NaN,public,26838050.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3068126,Picea mariana,whole plant height,12.192,m,NaN,NaN,NaN,NaN,NaN,Greg Reams,NaN,public,2024156.0
3068127,Quercus rubra,whole plant height,13.716,m,NaN,NaN,NaN,NaN,NaN,Greg Reams,NaN,public,2024184.0
3068128,Quercus rubra,whole plant height,17.0688,m,NaN,NaN,NaN,NaN,NaN,Greg Reams,NaN,public,2024189.0
3068129,Quercus rubra,whole plant height,21.336,m,NaN,NaN,NaN,NaN,NaN,Greg Reams,NaN,public,2024193.0


In [38]:
# plant height df
df["trait_value"].map(type).value_counts()

trait_value
<class 'float'>    2883584
<class 'str'>       184547
Name: count, dtype: int64

In [49]:
# plant height df
h = trait_dfs["whole plant height"].copy()
h["scrubbed_species_binomial"] = h["scrubbed_species_binomial"].astype(str).str.strip().str.lower()
h["plant_height_m"] = pd.to_numeric(h["trait_value"], errors="coerce")
h = h.dropna(subset=["scrubbed_species_binomial", "plant_height_m"])

height_summary = h.groupby("scrubbed_species_binomial").agg(
    plant_height_m_mean=("plant_height_m", "mean"),
    plant_height_m_min=("plant_height_m", "min"),
    plant_height_m_max=("plant_height_m", "max"),
    unit=("unit", lambda x: "; ".join(x.dropna().unique())),
    project_pi=("project_pi", lambda x: "; ".join(x.dropna().unique())),
)

height_summary = height_summary.reset_index()

height_summary

,scrubbed_species_binomial,plant_height_m_mean,plant_height_m_min,plant_height_m_max,unit,project_pi
0,aesculus hippocastanum,23.800000,20.0000,39.0000,m,Michael Kleyer; Price CA
1,buxus sempervirens,2.250000,0.3000,5.0000,m,Michael Kleyer; Price CA
2,camellia japonica,11.000000,11.0000,11.0000,m,Price CA
3,capsicum annuum,0.126828,0.0092,0.6173,m,Milla R
4,caragana arborescens,3.500000,2.0000,6.0000,m,Michael Kleyer
...,...,...,...,...,...,...
199,vicia sativa,0.696687,0.0500,1.3870,m,Loughnan D; Michael Kleyer; DostÃ¡l P; Marx HE
200,vicia tenuifolia,0.655000,0.2700,1.5000,m,Michael Kleyer
201,viola palustris,0.085000,0.0500,0.1200,m,Michael Kleyer
202,ziziphus jujuba,12.954000,11.8872,14.0208,m,


In [40]:
height_summary.to_csv("bien_summary_plant_height.csv", index=False)

In [50]:
# leaf area dataframe
la = trait_dfs["leaf area"].copy()
la["scrubbed_species_binomial"] = la["scrubbed_species_binomial"].astype(str).str.strip().str.lower()
la["leaf_area_mm2"] = pd.to_numeric(la["trait_value"], errors="coerce")

la = la.dropna(subset=["scrubbed_species_binomial", "leaf_area_mm2"])

leaf_area_summary = la.groupby("scrubbed_species_binomial").agg(
    leaf_area_m2_mean=("leaf_area_mm2", "mean"),
    leaf_area_m2_min=("leaf_area_mm2", "min"),
    leaf_area_m2_max=("leaf_area_mm2", "max"),
    unit=("unit", lambda x: "; ".join(x.dropna().unique())),
    project_pi=("project_pi", lambda x: "; ".join(x.dropna().unique())),
)

leaf_area_summary = leaf_area_summary.reset_index()

leaf_area_summary

,scrubbed_species_binomial,leaf_area_m2_mean,leaf_area_m2_min,leaf_area_m2_max,unit,project_pi
0,aesculus hippocastanum,6528.700000,6331.60,6725.8,mm2,K. Thompson; Price CA
1,buxus sempervirens,141.960000,108.10,180.3,mm2,Price CA; K. Thompson
2,capsicum annuum,732.809091,165.60,2650.1,mm2,Milla R
3,caragana arborescens,1059.500000,1059.50,1059.5,mm2,D.K. Kunzmann
4,carex acutiformis,7035.000000,7035.00,7035.0,mm2,K. Thompson
...,...,...,...,...,...,...
131,veronica serpyllifolia,188.000000,188.00,188.0,mm2,K. Thompson; Marx HE
132,viburnum tinus,1799.540000,1435.80,2313.0,mm2,K. Thompson; de la Riva EG; Price CA
133,vicia sativa,664.981420,186.25,1393.0,mm2,D.K. Kunzmann; K. Thompson; Marx HE
134,vicia tenuifolia,1904.606667,1059.00,2413.0,mm2,K. Thompson; D.K. Kunzmann


In [42]:
leaf_area_summary.to_csv("bien_summary_leaf_area.csv", index=False)

In [52]:
# stem wood density dataframe
wd = trait_dfs["stem wood density"].copy()
wd["scrubbed_species_binomial"] = wd["scrubbed_species_binomial"].astype(str).str.strip().str.lower()
wd["stem_wood_density_g_cm3"] = pd.to_numeric(wd["trait_value"], errors="coerce")

wd = wd.dropna(subset=["scrubbed_species_binomial", "stem_wood_density_g_cm3"])

stem_wood_density_summary = wd.groupby("scrubbed_species_binomial").agg(
    stem_wood_density_g_cm3_mean=("stem_wood_density_g_cm3", "mean"),
    stem_wood_density_g_cm3_min=("stem_wood_density_g_cm3", "min"),
    stem_wood_density_g_cm3_max=("stem_wood_density_g_cm3", "max"),
    unit=("unit", lambda x: "; ".join(x.dropna().unique())),
    project_pi=("project_pi", lambda x: "; ".join(x.dropna().unique())),
)

stem_wood_density_summary = stem_wood_density_summary.reset_index()

stem_wood_density_summary

,scrubbed_species_binomial,stem_wood_density_g_cm3_mean,stem_wood_density_g_cm3_min,stem_wood_density_g_cm3_max,unit,project_pi
0,aegle marmelos,0.825614,0.771000,0.880,g.cm-3,"Zanne, A.E., Lopez-Gonzalez, G., Coomes, D.A...."
1,aesculus hippocastanum,0.500000,0.500000,0.500,g.cm-3,"Zanne, A.E., Lopez-Gonzalez, G., Coomes, D.A...."
2,angophora floribunda,0.725227,0.660000,0.758,g.cm-3,"Zanne, A.E., Lopez-Gonzalez, G., Coomes, D.A...."
3,aquilaria malaccensis,0.320000,0.320000,0.320,g.cm-3,"Zanne, A.E., Lopez-Gonzalez, G., Coomes, D.A...."
4,aquilaria sinensis,0.366079,0.344000,0.380,g.cm-3,"Zanne, A.E., Lopez-Gonzalez, G., Coomes, D.A...."
...,...,...,...,...,...,...
197,umbellularia californica,0.511667,0.510000,0.515,g.cm-3,"D.D. Ackerly; Zanne, A.E., Lopez-Gonzalez, G...."
198,vernicia montana,0.318497,0.315987,0.321,g.cm-3,"Zanne, A.E., Lopez-Gonzalez, G., Coomes, D.A...."
199,viburnum tinus,0.465000,0.460000,0.470,g.cm-3,de la Riva EG
200,xylocarpus granatum,0.591206,0.525000,0.663,g.cm-3,"Zanne, A.E., Lopez-Gonzalez, G., Coomes, D.A...."


In [55]:
stem_wood_density_summary.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 202 entries, 0 to 201
Data columns (total 6 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   scrubbed_species_binomial     202 non-null    object 
 1   stem_wood_density_g_cm3_mean  202 non-null    float64
 2   stem_wood_density_g_cm3_min   202 non-null    float64
 3   stem_wood_density_g_cm3_max   202 non-null    float64
 4   unit                          202 non-null    object 
 5   project_pi                    202 non-null    object 
dtypes: float64(3), object(3)
memory usage: 9.6+ KB


In [44]:
stem_wood_density_summary.to_csv("bien_summary_stem_wood_density.csv", index=False)

In [54]:
# whole plant sexual system dataset
ss = trait_dfs["whole plant sexual system"].copy()
ss["scrubbed_species_binomial"] = ss["scrubbed_species_binomial"].astype(str).str.strip().str.lower()

def most_frequent(x):
    vc = x.value_counts()
    return vc.index[0] if not vc.empty else pd.NA

sexual_system_summary = ss.groupby("scrubbed_species_binomial").agg(
    sexual_system=("trait_value", most_frequent),
    project_pi=("project_pi", lambda x: "; ".join(x.dropna().astype(str).unique())),
)

sexual_system_summary = sexual_system_summary.reset_index()

sexual_system_summary

,scrubbed_species_binomial,sexual_system,project_pi
0,bruguiera gymnorhiza,Hermaphrodite,Bezeng BS
1,calotropis procera,Hermaphrodite,Bezeng BS
2,casuarina cunninghamiana,Hermaphrodite,Bezeng BS
3,casuarina equisetifolia,Hermaphrodite,Bezeng BS
4,cereus jamacaru,Hermaphrodite,Bezeng BS
5,ceriops tagal,Hermaphrodite,Bezeng BS
6,delonix regia,Hermaphrodite,Bezeng BS
7,dovyalis caffra,Dioecious,Bezeng BS
8,ensete ventricosum,Hermaphrodite,Bezeng BS
9,eucalyptus camaldulensis,Hermaphrodite,Bezeng BS


In [46]:
sexual_system_summary.to_csv("bien_summary_sexual_system.csv", index=False)